In [ ]:
# Install necessary Python packages
# Use pip package manager to install the following libraries:
# - accelerate: Library for high-performance training of PyTorch and TensorFlow models.
# - bitsandbytes: (Note: Confirm the correct name and version as it's not a standard package)
# - transformers: Library developed by Hugging Face for Natural Language Processing tasks, offering pre-trained models and utilities.
# - trl: Textual Response Loop library for response generation, particularly useful in conversational agents.

!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 xformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 6.6 MB/s eta 0:00:00a 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 11.3 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 MB 11.0 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 99.1 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.9 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.2/218.2 MB 2.5 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 21.7 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 26.2 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 767.3 kB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 65.0 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 69.4 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.3 MB/s eta 0:00:00

   

In [ ]:

# Command: !nvidia-smi
# Description:
# This command is used to display the current GPU (NVIDIA) utilization and memory usage.
# It provides information about the GPUs installed on the system, including their model, memory usage, and processes utilizing them.
# Note:
# - This command is specific to systems with NVIDIA GPUs and requires the NVIDIA System Management Interface (SMI) to be installed.
# - It's commonly used to check GPU availability and utilization before running GPU-intensive tasks like deep learning training or inference.

!nvidia-smi

In [ ]:
import os
import torch
import logging

# Import necessary modules and functions from external libraries
from datasets import load_dataset  # Import function to load datasets
from transformers import (
    AutoModelForCausalLM,  # Import class for auto model for causal language modeling
    AutoTokenizer,  # Import class for auto tokenizer
    BitsAndBytesConfig,  # Import class for BitsAndBytes configuration
    HfArgumentParser,  # Import class for Hugging Face argument parser
    TrainingArguments,  # Import class for training arguments
    pipeline)  # Import function for pipeline creation

from peft import LoraConfig, PeftModel  # Import classes from the peft module
from trl import SFTTrainer  # Import SFTTrainer class from the trl module

In [ ]:
# Load Dataset
# Description:
# Load a dataset from the specified dataset name and split using the load_dataset function from the datasets module.
# The loaded dataset is stored in the variable named 'dataset'.

# Dataset Name:
dataset_name = "mlabonne/guanaco-llama2-1k"

#Load Data
dataset = load_dataset(dataset_name, split="train")
# Usage:
# The load_dataset function retrieves the specified dataset from the Hugging Face datasets hub.
# It's commonly used for loading various datasets for Natural Language Processing tasks.
# Parameters:
# - dataset_name: The name or identifier of the dataset to load.
# - split: The split of the dataset to load. Common splits include "train", "validation", and "test".
# Return:
# The loaded dataset object, which contains the data as per the specified split.



In [ ]:
dataset

In [ ]:
# Pre-trained Model Name and Fine-tuned Model Name
# Description:
# These variables store the names of the pre-trained model and the fine-tuned model respectively.

# Pre-trained Model Name:
base_model_name = "NousResearch/Llama-2-7b-chat-hf"
# Description:
# The 'base_model_name' variable stores the name or identifier of the pre-trained model.
# This model serves as the base architecture that is fine-tuned for specific tasks.
# In this case, the model is named "NousResearch/Llama-2-7b-chat-hf".

# Fine-tuned Model Name:
fine_tuned_model = "llama-2-7b-finetuned-oa-1000"
# Description:
# The 'fine_tuned_model' variable stores the name or identifier of the fine-tuned model.
# This model is derived from the pre-trained model after fine-tuning it on specific data or tasks.
# In this case, the model is named "llama-2-7b-finetuned-oa-1000".




In [ ]:
# Tokenizer
# Description:
# The 'llama_tokenizer' variable represents an instance of the tokenizer used for tokenizing text data.
# It is initialized using the AutoTokenizer class from the transformers module.

# Initialization:
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
# Here, the AutoTokenizer.from_pretrained method is used to instantiate the tokenizer.
# - 'from_pretrained' loads the pre-trained tokenizer associated with the specified model name ('base_model_name').
# - 'base_model_name' is the pre-trained model name used to identify the model architecture and associated tokenizer.
# - 'trust_remote_code=True' enables the tokenizer to trust the remote code for downloading resources securely.

# Configuration:
# - 'pad_token': Set the padding token to be the end-of-sequence (eos) token of the tokenizer.
llama_tokenizer.pad_token = llama_tokenizer.eos_token
# - 'padding_side': Set the padding side to "right", indicating that padding will be added to the right side of sequences.
llama_tokenizer.padding_side = "right"



In [ ]:
# Quantization Config
# Description:
# The 'quant_config' variable stores the configuration settings for quantization.
# Quantization is a technique used to reduce the precision of the neural network's parameters and activations,
# which helps in reducing memory footprint and computational requirements while maintaining performance.

# Initialization:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load the model with 4-bit quantization
    bnb_4bit_quant_type="nf4",  # Use non-float 4-bit quantization
    bnb_4bit_compute_dtype=torch.float16,  # Set the compute data type to 16-bit floating point
    bnb_4bit_use_double_quant=True  # Disable double quantization
)

# Parameters:
# - 'load_in_4bit': Whether to load the model with 4-bit quantization enabled.
# - 'bnb_4bit_quant_type': Type of 4-bit quantization to be used (e.g., "nf4" for non-float 4-bit quantization).
# - 'bnb_4bit_compute_dtype': Compute data type used during 4-bit quantization (e.g., 16-bit floating point).
# - 'bnb_4bit_use_double_quant': Whether to use double quantization (True or False).


In [ ]:
# Load the Pre-trained Model
# Description:
# This code snippet loads the pre-trained model for causal language modeling using the AutoModelForCausalLM class from the transformers module.
# It initializes the model with the specified base model name and quantization configuration.

# Initialization:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,  # Name of the pre-trained model to load
    quantization_config=quant_config,  # Quantization configuration for the model
    device_map={"": 0}  # Device mapping for model parallelism (if applicable)
)

# Configuration:
# - 'base_model_name': Name or identifier of the pre-trained model to load.
# - 'quantization_config': Configuration settings for quantization, defined earlier.
# - 'device_map': Mapping of model components to specific devices for model parallelism.

# Additional Configuration:
# - 'use_cache': Set the use_cache attribute of the model configuration to False,
#   indicating that caching of the model's internal states during inference should be disabled.
base_model.config.use_cache = False

# - 'pretraining_tp': Set the pretraining_tp attribute of the model configuration to 1,
#   specifying the number of pre-training steps (if applicable).
base_model.config.pretraining_tp = 1

# Usage:
# The 'from_pretrained' method initializes the model with the specified pre-trained weights and configurations.
# It automatically downloads the pre-trained model if it is not available locally.

# Note:
# Ensure that the specified pre-trained model name is correct and accessible.
# Verify that the quantization configuration aligns with the model's architecture and hardware requirements.
# The additional configuration parameters ('use_cache' and 'pretraining_tp') may vary based on the model's specifications.


In [ ]:
base_model

In [ ]:
# LoRA Config
# Description:
# The 'peft_parameters' variable stores the configuration settings for LoRA (Locally Re-parameterized Attention).
# LoRA is a type of attention mechanism used in neural networks, particularly in Transformer architectures.

# Initialization:
peft_parameters = LoraConfig(
    lora_alpha=16,  # Alpha value for LoRA, controlling the degree of re-parameterization
    lora_dropout=0.1,  # Dropout rate applied to LoRA weights
    #target_modules = ["q_proj", "v_proj"],  # List of target modules for applying LoRA
    r=8,  # Number of attention heads
    bias="none",  # Bias type for LoRA
    task_type="CAUSAL_LM"  # Task type (e.g., "CAUSAL_LM" for causal language modeling)
)

# Parameters:
# - 'lora_alpha': Alpha value for LoRA, controlling the degree of re-parameterization.
# - 'lora_dropout': Dropout rate applied to LoRA weights to prevent overfitting.
# - 'r': Number of attention heads in the attention mechanism.
# - 'bias': Type of bias applied to LoRA (e.g., "none", "qk", "v").
# - 'task_type': Type of task for which the model is designed (e.g., "CAUSAL_LM" for causal language modeling).

# Usage:
# The 'peft_parameters' object encapsulates the configuration settings for LoRA.
# It can be used to configure LoRA-based attention mechanisms within Transformer models.

# Example:
# You can pass the 'peft_parameters' object as an argument when initializing a model that uses LoRA attention.
# Example:
# model_with_lora = ModelWithLoRA(config=peft_parameters)

# Note:
# Ensure that the configuration settings align with the model's architecture and task requirements.
# LoRA is a specialized attention mechanism and may require specific tuning based on the model's characteristics.


In [ ]:
# Training Params
# Description:
# The 'train_params' variable stores the training configuration parameters used during model training.
# These parameters control various aspects of the training process, such as batch size, optimization algorithm, learning rate, etc.

# Initialization:
train_params = TrainingArguments(
    output_dir="./results_modified",  # Output directory to save the training results
    num_train_epochs=1,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size per device for training
    gradient_accumulation_steps=1,  # Number of gradient accumulation steps
    optim="paged_adamw_32bit",  # Optimization algorithm used for training
    save_steps=25,  # Frequency of saving model checkpoints during training
    logging_steps=25,  # Frequency of logging training metrics
    learning_rate=3e-4,  # Initial learning rate for the optimizer
    weight_decay=0.001,  # Weight decay rate for regularization
    fp16=False,  # Whether to use 16-bit floating point precision (FP16)
    bf16=False,  # Whether to use 16-bit brain floating point precision (BF16)
    max_grad_norm=0.3,  # Maximum gradient norm for gradient clipping
    max_steps=-1,  # Maximum number of training steps (-1 indicates no limit)
    warmup_ratio=0.01,  # Ratio of warmup steps to total training steps
    group_by_length=True,  # Whether to group samples by sequence length for efficient batching
    lr_scheduler_type="constant"  # Type of learning rate scheduler (constant, linear, etc.)
)

# Parameters:
# - 'output_dir': Directory path to save the training results, including model checkpoints and logs.
# - 'num_train_epochs': Number of training epochs to iterate over the dataset.
# - 'per_device_train_batch_size': Batch size per device used for training.
# - 'gradient_accumulation_steps': Number of gradient accumulation steps before updating model parameters.
# - 'optim': Optimization algorithm used for training (e.g., "paged_adamw_32bit").
# - 'save_steps': Frequency of saving model checkpoints during training.
# - 'logging_steps': Frequency of logging training metrics.
# - 'learning_rate': Initial learning rate for the optimizer.
# - 'weight_decay': Weight decay rate for L2 regularization.
# - 'fp16': Whether to use 16-bit floating point precision (FP16) for training.
# - 'bf16': Whether to use 16-bit brain floating point precision (BF16) for training.
# - 'max_grad_norm': Maximum gradient norm for gradient clipping to prevent exploding gradients.
# - 'max_steps': Maximum number of training steps (-1 indicates no limit).
# - 'warmup_ratio': Ratio of warmup steps to total training steps for learning rate warmup.
# - 'group_by_length': Whether to group samples by sequence length for efficient batching.
# - 'lr_scheduler_type': Type of learning rate scheduler used (constant, linear, etc.).

# Usage:
# The 'train_params' object encapsulates the training configuration parameters.
# It is used to configure the training process when initializing the Trainer object for model training.

# Example:
# trainer = Trainer(model, args=train_params)

# Note:
# Ensure that the training parameters are set according to the requirements of the model architecture and the dataset.
# Fine-tuning these parameters may be necessary to achieve optimal training performance.


In [ ]:
# Trainer to train the model
# Description:
# The 'fine_tuning' variable represents the trainer object responsible for fine-tuning the base model on the provided dataset.
# It utilizes the SFTTrainer class from the trl module for training.

# Initialization:
fine_tuning = SFTTrainer(
    model=base_model,  # Pre-trained model to be fine-tuned
    train_dataset=dataset,  # Training dataset for fine-tuning
    peft_config=peft_parameters,  # Configuration parameters for Peft (if applicable)
    dataset_text_field="text",  # Field in the dataset containing text data
    tokenizer=llama_tokenizer,  # Tokenizer for tokenizing input text
    args=train_params  # Training configuration parameters
)

# Parameters:
# - 'model': Pre-trained model to be fine-tuned on the dataset.
# - 'train_dataset': Training dataset used for fine-tuning the model.
# - 'peft_config': Configuration parameters for Peft (if applicable, otherwise can be omitted).
# - 'dataset_text_field': Field in the dataset containing text data to be used for training.
# - 'tokenizer': Tokenizer object for tokenizing input text data.
# - 'args': Training configuration parameters controlling various aspects of the training process.

# Usage:
# The 'fine_tuning' trainer object encapsulates the fine-tuning process for the model.
# It is responsible for iterating over the training dataset, computing gradients, and updating model parameters.

# Example:
# fine_tuning.train()
# This line initiates the training process, causing the model to be fine-tuned on the provided dataset.

# Note:
# Ensure that the model, dataset, tokenizer, and training parameters are properly configured before initiating training.
# Fine-tuning a model requires careful consideration of hyperparameters and dataset characteristics to achieve optimal performance.


In [ ]:
# Training starts here
# Description:
# This line initiates the training process using the 'fine_tuning' trainer object.
# It triggers the fine-tuning of the pre-trained model on the provided dataset using the specified training configuration.

# Usage:
fine_tuning.train()
# This method call starts the training process, causing the model to be fine-tuned on the provided dataset.
# During training, the model iterates over the dataset, computes gradients, and updates model parameters based on the optimization algorithm.

# Note:
# Ensure that the trainer object ('fine_tuning') and its associated components (model, dataset, tokenizer, etc.) are properly configured before initiating training.
# Training a model may take a significant amount of time depending on the dataset size, model complexity, and hardware specifications.
# Monitor training progress, evaluate model performance, and adjust training parameters as needed to achieve desired results.


In [ ]:
# Save the fine-tuned Model
# Description:
# This line saves the fine-tuned model to the specified directory or file path.
# It uses the save_pretrained method available for the model object to save the model's weights and configuration.

# Usage:
fine_tuning.model.save_pretrained(fine_tuned_model)
# This method call saves the fine-tuned model to the directory or file path specified by 'fine_tuned_model'.
# The saved model includes the model's weights, configuration, and any additional artifacts required for reloading the model later.

# Note:
# Ensure that the specified directory or file path is accessible and has write permissions.
# Saving the fine-tuned model allows you to reuse it for inference, evaluation, or further fine-tuning in future tasks.
# It's recommended to save models along with their associated configurations and tokenizer for reproducibility and ease of use.

In [ ]:
# from transformers import (
#     AutoTokenizer,
#     pipeline)

In [ ]:
# base_model_name = "NousResearch/Llama-2-7b-chat-hf"
# #fine_tuned_model = "llama-2-7b-finetuned"
# fine_tuned_model = "llama-2-7b-finetuned-oa-1000"

In [ ]:
# llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
# llama_tokenizer.pad_token = llama_tokenizer.eos_token
# llama_tokenizer.padding_side = "right"




In [ ]:
from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig

In [ ]:
# how to load peft model from hub for inference
config = PeftConfig.from_pretrained(fine_tuned_model)

In [ ]:
from transformers import AutoModelForCausalLM

# Load the pre-trained model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,  # Name of the pre-trained model to load
    quantization_config=quant_config,  # Quantization configuration for the model
    device_map={"": 0}  # Device mapping for model parallelism (if applicable)
)

# Disable caching of model internal states during inference
base_model.config.use_cache = False

# Set the number of pretraining steps to 1
base_model.config.pretraining_tp = 1


In [ ]:
# let's generate some text
query = "best way to write a leave letter for student?"
text_gen = pipeline(task="text-generation", model=base_model, tokenizer=llama_tokenizer, max_length=200)
output = text_gen(f"<s>[INST] {query} [/INST]")
print(output[0]['generated_text'])

In [ ]:

#llama_tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
lora_model = PeftModel.from_pretrained(base_model, fine_tuned_model)

In [ ]:
# let's generate some text from the fine-tuned model
query = "best way to write a leave letter for student?"
text_gen = pipeline(task="text-generation", model=lora_model, tokenizer=llama_tokenizer, max_length=200)
output = text_gen(f"<s>[INST] {query} [/INST]")
print(output[0]['generated_text'])